# Methylation Branch Remediation - Results Visualization

Charts summarizing the diagnostic investigation and fix documented in `MethylationRemediation.ipynb`: why the original 1D-CNN CV cell collapsed to chance-level performance, and how the fixed 1D-CNN (frozen pan-cancer backbone + fixed per-fold PCA(50) + small trainable Dense head) resolves it.

All numbers below are taken directly from the executed runs in `MethylationRemediation.ipynb` and the standalone `code/scripts/step1-4*.py` diagnostic scripts. Charts are built with Plotly.

In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLOR_BAD = '#d9534f'
COLOR_GOOD = '#2e7d32'
COLOR_MID = '#f0ad4e'
COLOR_REF = '#9c27b0'


## 1. Headline result: original CNN head vs. fixed CNN head

The single most important chart: the original notebook's CV cell collapsed to chance on every metric, while the fixed 1D-CNN head (same frozen pan-cancer backbone, PCA-reduced input, properly-sized head) clears it by a wide margin on all three metrics.

In [2]:
metrics = ['MCC', 'AUC', 'fvPTC Recall']
original = [-0.0018, 0.5169, 0.3951]
fixed = [0.50, 0.8252, 0.6831]   # from the executed final CV cell in MethylationRemediation.ipynb

fig = go.Figure()
fig.add_bar(name='Original CNN head (chance)', x=metrics, y=original, marker_color=COLOR_BAD,
            text=[f'{v:.3f}' for v in original], textposition='outside')
fig.add_bar(name='Fixed CNN head', x=metrics, y=fixed, marker_color=COLOR_GOOD,
            text=[f'{v:.3f}' for v in fixed], textposition='outside')
fig.add_hline(y=0.5, line_dash='dash', line_color='gray',
              annotation_text='Chance level (AUC=0.5 / MCC=0)', annotation_position='top left')

fig.update_layout(
    title='Methylation Branch: Original vs. Fixed 1D-CNN Head<br><sup>5-fold Stratified CV, held-out folds</sup>',
    barmode='group', yaxis_title='Score', yaxis_range=[-0.15, 1.0],
    template='plotly_white', width=700, height=480,
)
fig.show()


## 2. Per-fold MCC across the diagnostic sequence

Shows why the original CNN head failed and how each diagnostic step ruled hypotheses in or out. "Original CNN head" comes from the notebook's own reported per-fold results; the rest are from this investigation's diagnostic scripts, all using the identical `StratifiedKFold(n_splits=5, random_state=42)` protocol.

In [3]:
fold_labels = ['Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5']

series = {
    'Original CNN head (chance, collapses)': ([0.0, 0.0, 0.0, 0.0, 0.0], COLOR_BAD),
    'QDA, raw 485k probes (Step 1)': ([-0.1262, -0.0803, 0.0648, -0.0879, 0.0042], COLOR_MID),
    'SVC, raw 485k probes (Step 1)': ([-0.0615, 0.1071, 0.2957, 0.0624, 0.1276], COLOR_MID),
    'SVC, frozen embedding + PCA(50) (Step 2/3)': ([0.5271, 0.6050, 0.4633, 0.6161, 0.4680], COLOR_REF),
    'Fixed CNN head (Step 4, final)': ([0.5477, 0.5140, 0.5113, 0.6865, 0.4132], COLOR_GOOD),
}

fig = go.Figure()
for label, (vals, color) in series.items():
    fig.add_bar(name=label, x=fold_labels, y=vals, marker_color=color)

fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(
    title='Per-fold MCC across the diagnostic sequence',
    barmode='group', yaxis_title='MCC (held-out fold)',
    template='plotly_white', width=900, height=520,
    legend=dict(orientation='h', yanchor='bottom', y=-0.35, xanchor='center', x=0.5, font=dict(size=9)),
)
fig.show()


## 3. Fold-collapse signature: original vs. fixed

The original head's defining failure wasn't "weak performance" -- it was *exact* 0.0 MCC in every fold because it always predicted a single constant class (folds 1-3 always cvPTC, folds 4-5 always fvPTC). This chart shows fvPTC recall per fold for both models: 0.0 or 1.0 marks a full collapse to one class.

In [4]:
original_recall1 = [0.0, 0.0, 0.0, 1.0, 1.0]      # from the original notebook's CV cell output
fixed_recall1 = [0.8125, 0.6471, 0.7059, 0.9375, 0.5625]  # step4 sweep winner, per-fold

fig = make_subplots(rows=1, cols=2, subplot_titles=('Original CNN head', 'Fixed CNN head'), shared_yaxes=True)

fig.add_bar(x=fold_labels, y=original_recall1, marker_color=COLOR_BAD,
            text=[f'{v:.2f}' + (' (collapsed)' if v in (0.0, 1.0) else '') for v in original_recall1],
            textposition='outside', row=1, col=1, showlegend=False)
fig.add_bar(x=fold_labels, y=fixed_recall1, marker_color=COLOR_GOOD,
            text=[f'{v:.2f}' for v in fixed_recall1], textposition='outside', row=1, col=2, showlegend=False)

fig.update_yaxes(range=[-0.05, 1.25], title_text='fvPTC recall', row=1, col=1)
fig.update_yaxes(range=[-0.05, 1.25], row=1, col=2)
fig.update_layout(
    title='fvPTC recall per fold: original head collapses to 0 or 1 every time, fixed head does not',
    template='plotly_white', width=900, height=480,
)
fig.show()


## 4. Hyperparameter sweep results (Step 4)

Every configuration tried for the fixed 1D-CNN head clears the original chance-level baseline by a wide margin, showing the fix is robust to the exact head size/regularization choice, not a single lucky configuration.

In [5]:
sweep_configs = [
    'pca=50, hid=16<br>l2=.01 do=.3',
    'pca=50, hid=32<br>l2=.01 do=.4',
    'pca=30, hid=16<br>l2=.01 do=.3',
    'pca=50, hid=16<br>l2=.05 do=.5',
    'pca=20, hid=8<br>l2=.01 do=.3',
]
sweep_mcc = [0.5874, 0.4521, 0.4846, 0.5346, 0.4883]  # step4 standalone sweep run

fig = go.Figure()
fig.add_bar(x=sweep_configs, y=sweep_mcc, marker_color=COLOR_GOOD,
            text=[f'{v:.3f}' for v in sweep_mcc], textposition='outside')
fig.add_hline(y=0.0, line_dash='dash', line_color=COLOR_BAD,
              annotation_text='Original CNN head (MCC = -0.0018)', annotation_position='bottom right')

fig.update_layout(
    title='Step 4 hyperparameter sweep: every configuration beats the original head',
    yaxis_title='Aggregate CV MCC', template='plotly_white', width=800, height=500,
)
fig.show()


## 5. Aggregate confusion matrix: fixed model (best sweep config, all 5 folds pooled)

Pooling predictions across all 5 held-out folds gives a single confusion matrix summarizing the fixed model's real-world error pattern on unseen data.

In [6]:
import numpy as np

# Pooled from the 5 per-fold confusion matrices printed in step3/step4 runs
# (cvPTC=0, fvPTC=1), best sweep config (pca=50, hidden=16, l2=.01, dropout=.3, lr=1e-3)
cm_pooled = np.array([
    [46 + 48 + 48 + 47 + 47, 12 + 9 + 9 + 10 + 10],
    [3 + 3 + 6 + 2 + 5,       13 + 14 + 11 + 14 + 11],
])
labels = ['cvPTC', 'fvPTC']

fig = go.Figure(data=go.Heatmap(
    z=cm_pooled, x=[f'Pred {l}' for l in labels], y=[f'True {l}' for l in labels],
    colorscale='Greens', showscale=False, text=cm_pooled, texttemplate='%{text}', textfont_size=16,
))
fig.update_layout(
    title='Fixed 1D-CNN head: pooled confusion matrix<br><sup>5-fold CV, 368 held-out predictions</sup>',
    template='plotly_white', width=500, height=480, yaxis_autorange='reversed',
)
fig.show()

tp = cm_pooled[1, 1]; fn = cm_pooled[1, 0]; tn = cm_pooled[0, 0]; fp = cm_pooled[0, 1]
print(f'cvPTC recall: {tn/(tn+fp):.3f}   fvPTC recall: {tp/(tp+fn):.3f}')
print(f'Overall accuracy: {(tp+tn)/cm_pooled.sum():.3f}')


cvPTC recall: 0.825   fvPTC recall: 0.768
Overall accuracy: 0.812


## Summary

- The headline chart (Section 1) shows the fixed 1D-CNN head clears chance level on every metric that the original head failed on.
- The per-fold view (Section 2) shows the fix is not a fluke of one lucky split -- MCC is positive and stable in every fold, unlike the original head's exact 0.0 in all five.
- The fold-collapse chart (Section 3) makes the *actual* bug visible directly: the original head didn't perform "a bit worse," it degenerated into a constant-class predictor every single time.
- The sweep chart (Section 4) shows the fix is robust across a range of head sizes/regularization settings, not a single cherry-picked configuration.
- The pooled confusion matrix (Section 5) gives a single, honest picture of the fixed model's real error pattern across all 368 held-out CV predictions.
